# Pinecone Uploader

`result.csv`를 읽어 `blob` 컬럼의 embedding vector를 복원한 뒤 Pinecone에 업로드합니다.

- vector 값: `blob` 컬럼을 Base64 decode 후 `float32` 배열로 복원
- metadata: `blob`을 제외한 모든 컬럼을 `{컬럼명: 값}` 형태로 저장
- id: `feature_index` 형식으로 자동 생성

In [ ]:
import base64
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from pinecone import Pinecone
from tqdm.auto import tqdm

In [ ]:
# 1. 기본 설정
BASE_DIR = Path.cwd()
RESULT_CSV = BASE_DIR / "result.csv"

BLOB_COLUMN = "blob"
UPSERT_BATCH_SIZE = 100

load_dotenv(BASE_DIR / ".env")
NAMESPACE = os.getenv("PINECONE_NAMESPACE", "user_manual")

pinecone_api_key = os.getenv("PINECONE_API_KEY")
pinecone_host = os.getenv("PINECONE_HOST")

if not pinecone_api_key:
    raise ValueError("PINECONE_API_KEY가 .env에 없습니다.")
if not pinecone_host:
    raise ValueError("PINECONE_HOST가 .env에 없습니다.")

pc = Pinecone(api_key=pinecone_api_key)
index = pc.Index(host=pinecone_host)

In [ ]:
# 2. result.csv 로드
df = pd.read_csv(RESULT_CSV)

required_columns = {"content", "feature", "index", BLOB_COLUMN}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"result.csv에 필수 컬럼이 없습니다: {sorted(missing_columns)}")

df = df.dropna(subset=[BLOB_COLUMN]).reset_index(drop=True)
print(f"업로드 대상 데이터 개수: {len(df)}개")
df.head()

In [ ]:
# 3. BLOB 복원 및 metadata 변환 함수
def parse_blob(encoded: str) -> np.ndarray:
    blob = base64.b64decode(encoded)
    return np.frombuffer(blob, dtype=np.float32)


def clean_metadata_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    return value


def make_vector(row: pd.Series) -> dict:
    metadata = {
        column: clean_metadata_value(row[column])
        for column in df.columns
        if column != BLOB_COLUMN
    }
    metadata = {key: value for key, value in metadata.items() if value is not None}

    vector_id = f"{row['feature']}_{row['index']}"

    return {
        "id": str(vector_id),
        "values": parse_blob(row[BLOB_COLUMN]).tolist(),
        "metadata": metadata,
    }

In [ ]:
# 4. 첫 번째 row 변환 테스트
sample_vector = make_vector(df.iloc[0])

print("ID:")
print(sample_vector["id"])

print("\nMetadata:")
print(sample_vector["metadata"])

print("\nVector 앞 5개:")
print(sample_vector["values"][:5])

print("\nVector dimension:")
print(len(sample_vector["values"]))

In [ ]:
# 5. Pinecone 업로드
vectors = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="vector 생성 중"):
    try:
        vectors.append(make_vector(row))
    except Exception as e:
        print(f"{idx} 번째 row 변환 에러")
        print(row.drop(labels=[BLOB_COLUMN], errors="ignore").to_dict())
        raise e

for i in tqdm(range(0, len(vectors), UPSERT_BATCH_SIZE), desc="Pinecone 업로드 중"):
    batch = vectors[i : i + UPSERT_BATCH_SIZE]
    index.upsert(vectors=batch, namespace=NAMESPACE)

print(f"업로드 완료: {len(vectors)}개, namespace={NAMESPACE}")